# WeatherNext Mini — Dayli AI Forecast Runner

Runs **WeatherNextCyclones_Mini** (Google DeepMind, Apache 2.0, 1.0°) and writes a
schema-v1 forecast JSON for Dayli's `auroraFeed.js`.

## Google Colab setup (recommended)

1. **Runtime → Change runtime type → v5e-1 TPU**
2. **Runtime → Run all.** No restart is needed.

## Kaggle setup

1. **Settings → Accelerator → any TPU option** (the chip type is detected)
2. **Settings → Internet → On**
3. **Run All.**

## How it works

The model runs in its own Python 3.12 environment in `/tmp/wn-env`, built by
cell 1, as a subprocess. The notebook never installs into the platform's own
Python and never imports jax itself, so the platform's packages can't conflict
with the model's and the TPU stays free for the subprocess.

If you restart the kernel, just re-run cell 1. It skips the install when the
environment already exists.

## Data note

Uses Google's **sample ECMWF HRES data from 2024-10-07**. The output is a real
model forecast, but not a current one. Live forecasts for the chatbot come from
`run_aifs_openmeteo.py`.

## 1. Build the isolated environment (~3-5 min, first run only)

In [ ]:
import os, subprocess, sys

ENV = '/tmp/wn-env'
PY = f'{ENV}/bin/python'
MARKER = f'{ENV}/.dayli-ready'

# Strip Kaggle's pip pins and its PYTHONPATH so nothing from the system
# Python 3.13 leaks into the isolated environment.
CLEAN_ENV = {k: v for k, v in os.environ.items()
             if k not in ('PIP_CONSTRAINT', 'PYTHONPATH', 'PYTHONHOME')}
CLEAN_ENV['PIP_CONFIG_FILE'] = os.devnull

import glob

ON_KAGGLE = os.path.isdir('/kaggle/working') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

# Set (e.g. 'v5litepod-8', 'v4-8') only if Kaggle's detection prints a wrong type.
TPU_TYPE_OVERRIDE = ''


def configure_kaggle_tpu():
    # libtpu learns the TPU type and topology from the GCE metadata server.
    # Colab exposes it; Kaggle does not. So on Kaggle only, read the chips off
    # the PCI bus and set the topology explicitly.
    pci = {'0x005e': ('v4', 2), '0x0062': ('v5p', 2),
           '0x0063': ('v5litepod', 1), '0x006f': ('v6e', 1)}
    chips = []
    for dev in glob.glob('/sys/bus/pci/devices/*'):
        try:
            if open(f'{dev}/vendor').read().strip() == '0x1ae0':
                did = open(f'{dev}/device').read().strip()
                if did in pci:
                    chips.append(pci[did])
        except OSError:
            pass
    if chips:
        gen, cores, n = chips[0][0], chips[0][1], len(chips)
    elif glob.glob('/dev/accel*'):
        gen, cores, n = 'v4', 2, len(glob.glob('/dev/accel*'))
    else:
        gen, cores, n = 'v5litepod', 1, len(glob.glob('/dev/vfio/[0-9]*'))
    if n == 0:
        raise SystemExit('No TPU chips found. Set Settings -> Accelerator to a TPU option.')
    tpu_env = {
        'TPU_ACCELERATOR_TYPE': TPU_TYPE_OVERRIDE or f'{gen}-{n * cores}',
        'TPU_CHIPS_PER_HOST_BOUNDS': {1: '1,1,1', 4: '2,2,1', 8: '2,4,1'}.get(n, f'{n},1,1'),
        'TPU_HOST_BOUNDS': '1,1,1',
        'TPU_WORKER_ID': '0',
        'TPU_WORKER_HOSTNAMES': 'localhost',
        'TPU_SKIP_MDS_QUERY': '1',
    }
    for k, v in tpu_env.items():
        # jax writes its warning text into these vars when the metadata lookup
        # fails, so treat that as unset.
        if not CLEAN_ENV.get(k) or CLEAN_ENV[k].startswith('WARNING'):
            CLEAN_ENV[k] = v
    print(f'Kaggle TPU: {n} chip(s) -> {CLEAN_ENV["TPU_ACCELERATOR_TYPE"]}')


if ON_KAGGLE:
    configure_kaggle_tpu()
else:
    print('Not on Kaggle - libtpu will detect the TPU itself')


def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=CLEAN_ENV)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'command failed with exit code {p.returncode}')


if os.path.exists(MARKER):
    print('environment already built, skipping install')
else:
    # uv downloads a standalone Python 3.12, independent of Kaggle's 3.13.
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    run([sys.executable, '-m', 'uv', 'venv', ENV, '--python', '3.12', '--seed', '--clear'])
    run([PY, '-m', 'pip', 'install', '--no-cache-dir', '--ignore-requires-python',
         'https://github.com/google-deepmind/weathernext/archive/master.zip',
         'jax[tpu]', 'h5netcdf',
         '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html'])
    open(MARKER, 'w').close()
    print('\nenvironment ready')

## 2. Verify the environment and TPU

In [ ]:
# Imports everything the model needs inside the isolated env and checks the TPU.
CHECK = '''
import sys
import numpy, scipy, jax, haiku, xarray
from scipy.spatial import transform
from weathernext.weathernext2 import fgn, architecture
print('python ', sys.version.split()[0])
print('numpy  ', numpy.__version__)
print('jax    ', jax.__version__)
print('haiku  ', haiku.__version__)
print('backend', jax.default_backend())
print('devices', jax.local_device_count(), [d.device_kind for d in jax.devices()][:1])
if jax.default_backend() != 'tpu':
    sys.exit('TPU not visible to JAX - see the libtpu messages above, and check '
             'the "TPU detected" line printed by cell 1')
print('OK - environment is consistent and the TPU is visible')
'''
run([PY, '-c', CHECK])

## 3. Write the inference script

In [ ]:
%%writefile /tmp/wn_infer.py
# Runs inside the isolated env (/tmp/wn-env), never in the notebook kernel.
import dataclasses
import json
import os
import sys
import time
import urllib.parse
import urllib.request

import haiku as hk
import jax
import numpy as np
import xarray
import xarray_jax
from weathernext.utils import checkpoint, data_utils, fiddle_config_io, rollout
from weathernext.weathernext2 import fgn

MODEL = 'WeatherNextCyclones_Mini'
SPLIT = '2024'
RES = '1.0'
STEPS = '20'  # 20 x 6h, the sample file's horizon
BUCKET = 'https://storage.googleapis.com/dm_graphcast/'
DATA_DIR = '/tmp/wn-data'
# Same box as run_aifs_openmeteo.py. All-India would exceed the server's
# 100 KB request limit; this box is what the chatbot uses.
REGION = dict(lat_min=16.0, lat_max=19.0, lon_min=77.0, lon_max=80.0)
HYD = (17.385, 78.487)
OUT = sys.argv[1] if len(sys.argv) > 1 else '/tmp/weathernext_india_latest.json'


def log(msg):
    print(f'[wn] {msg}', flush=True)


def fetch(path):
    # Public bucket, so plain HTTPS works without google-cloud-storage.
    os.makedirs(DATA_DIR, exist_ok=True)
    name = path.rsplit('/', 1)[-1]
    for a, b in ((' ', '_'), (':', '-'), ('<', 'lt')):
        name = name.replace(a, b)
    local = os.path.join(DATA_DIR, name)
    if os.path.exists(local) and os.path.getsize(local) > 0:
        log(f'cached {name}')
        return local
    log(f'downloading {name} ...')
    t0 = time.time()
    urllib.request.urlretrieve(BUCKET + urllib.parse.quote(path), local + '.part')
    os.replace(local + '.part', local)
    log(f'  {os.path.getsize(local) / 1e6:.0f} MB in {time.time() - t0:.0f}s')
    return local


backend = jax.default_backend()
n_dev = jax.local_device_count()
log(f'jax {jax.__version__}, backend={backend}, devices={n_dev}')
if backend != 'tpu':
    sys.exit('TPU not visible to JAX - re-run cells 1 and 2 and check their output')

data_file = fetch(
    'weathernext2/dataset/source-hres_forecast_init-2024-10-07 00:00:00'
    f'_res-{RES}_levels-13_steps-{STEPS}.nc')
weights_file = fetch(f'weathernext2/params/{MODEL}_<{SPLIT}.npz')

example_batch = xarray.load_dataset(data_file).compute()
config = fiddle_config_io.get_fiddle_config_by_name(f'weathernext2/configs/{MODEL}')
with open(weights_file, 'rb') as f:
    ckpt = checkpoint.load(f, fgn.CheckPoint)
log('weights, config and data loaded')

# Block sizes from Google's demo, tuned for v5e chips. v5e-8 is 8 of them.
transformer_kwargs = config.predictor_kwargs['noisy_function_kwargs'][
    'mesh_model_ctor'].keywords['transformer_kwargs']
transformer_kwargs.update({k: 128 for k in (
    'block_q', 'block_kv', 'block_kv_compute',
    'block_q_dkv', 'block_kv_dkv', 'block_kv_dkv_compute')})

config_inference = fgn.PredictorConfig(
    task=config.task,
    predictor_constructor=config.predictor_constructor,
    predictor_kwargs=config.predictor_kwargs,
    predictor_wrappers=config.predictor_wrappers[:-1],  # drop the ensemble wrapper
)


@hk.transform
def run_forward(inputs, targets_template, forcings):
    predictor = fgn.construct_predictor(config_inference)
    return predictor(inputs, targets_template=targets_template, forcings=forcings)


run_forward_jitted = jax.jit(
    lambda rng, i, t, f: run_forward.apply(ckpt.params, rng, i, t, f))
run_forward_pmap = xarray_jax.pmap(run_forward_jitted, dim='sample')

n_targets = (example_batch.sizes['time'] - 2) * 6
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch,
    target_lead_times=slice('6h', f'{n_targets}h'),
    **dataclasses.asdict(config.task))

# The rollout pmaps over devices, so the member count must be a multiple of the
# device count. One member per chip costs the same wall time as a single member.
members = n_dev
rng = jax.random.PRNGKey(0)
rngs = np.stack([jax.random.fold_in(rng, i) for i in range(members)], axis=0)
log(f'rolling out {eval_targets.sizes["time"]} steps x {members} members '
    '(step 1 includes XLA compilation and takes a few minutes)')

chunks = []
t0 = time.time()
for i, chunk in enumerate(rollout.chunked_prediction_generator_multiple_runs(
        predictor_fn=run_forward_pmap,
        rngs=rngs,
        inputs=eval_inputs,
        targets_template=eval_targets * np.nan,
        forcings=eval_forcings,
        num_steps_per_chunk=1,
        num_samples=members,
        pmap_devices=jax.local_devices())):
    chunks.append(chunk)
    log(f'  step {i + 1} ({time.time() - t0:.1f}s)')
    t0 = time.time()
predictions = xarray.combine_by_coords(chunks)
log(f'inference done: {dict(predictions.sizes)}')

# Ensemble mean. WeatherNext is generative, so single members are noisy.
pred = predictions.mean(dim='sample').isel(batch=0)
lat = pred.coords['lat'].values
lon = pred.coords['lon'].values
lat_m = (lat >= REGION['lat_min']) & (lat <= REGION['lat_max'])
lon_m = (lon >= REGION['lon_min']) & (lon <= REGION['lon_max'])
box = np.ix_(lat_m, lon_m)

init = np.datetime64(example_batch.isel(batch=0, time=1).datetime.values, 's')


def iso(t):
    return str(np.datetime_as_string(np.datetime64(t, 's'))) + 'Z'


def field(name, t_idx):
    return pred[name].isel(time=t_idx).transpose('lat', 'lon').values[box].astype(np.float64)


steps = []
for t_idx, lead in enumerate(pred.coords['time'].values):
    t2m = field('2m_temperature', t_idx) - 273.15
    u = field('10m_u_component_of_wind', t_idx)
    v = field('10m_v_component_of_wind', t_idx)
    wind = np.sqrt(u ** 2 + v ** 2) * 3.6
    if not (np.isfinite(t2m).all() and np.isfinite(wind).all()):
        sys.exit(f'non-finite model output at step {t_idx + 1}')
    steps.append({
        'valid_time': iso(init + lead),
        't2m_c': np.round(t2m, 2).tolist(),
        'wind10m_kph': np.round(wind, 2).tolist(),
    })

payload = {
    'schema_version': 1,
    'provenance': 'weathernext-inference',
    'model': f'Google WeatherNextCyclones_Mini 1.0deg, {members}-member ensemble mean, cropped to Telangana',
    'initial_conditions': 'ECMWF HRES forecast (GCS sample data, 2024-10-07 00Z)',
    'init_time': iso(init),
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'grid': {
        'lats': [round(float(x), 4) for x in lat[lat_m]],
        'lons': [round(float(x), 4) for x in lon[lon_m]],
    },
    'steps': steps,
}
with open(OUT, 'w') as f:
    json.dump(payload, f)
log(f'wrote {OUT} ({os.path.getsize(OUT) / 1024:.0f} KB, {len(steps)} steps)')

## 4. Run the model

In [ ]:
# Runs the model in the isolated env. Expect roughly 5-10 minutes the first
# time: about 1-2 min downloading, a few minutes of XLA compile, then the rollout.
OUT = '/tmp/weathernext_india_latest.json'
run([PY, '-u', '/tmp/wn_infer.py', OUT])

## 5. Check the Hyderabad forecast

In [ ]:
import json, os, shutil

OUT = '/tmp/weathernext_india_latest.json'
with open(OUT) as f:
    payload = json.load(f)

lats, lons = payload['grid']['lats'], payload['grid']['lons']
hi = min(range(len(lats)), key=lambda i: abs(lats[i] - 17.385))
hj = min(range(len(lons)), key=lambda j: abs(lons[j] - 78.487))

print('model:', payload['model'])
print('init :', payload['init_time'])
print(f'grid : {len(lats)} x {len(lons)}, {len(payload["steps"])} steps')
print(f'\nHyderabad (nearest cell {lats[hi]}N, {lons[hj]}E):')
temps = []
for s in payload['steps']:
    t, w = s['t2m_c'][hi][hj], s['wind10m_kph'][hi][hj]
    temps.append(t)
    print(f'  {s["valid_time"]}  {t:5.1f} C  {w:5.1f} km/h')

print(f'\nrange {min(temps):.1f} to {max(temps):.1f} C')
assert -10 < min(temps) and max(temps) < 55, 'temperatures are not physically plausible'
print('sanity check passed')

if os.path.isdir('/kaggle/working'):
    shutil.copy(OUT, '/kaggle/working/weathernext_india_latest.json')
    print('copied to /kaggle/working/ (download it from the Output panel)')
else:
    try:
        from google.colab import files
        files.download(OUT)  # browser download of the JSON
    except ImportError:
        print('saved at', OUT)

## 6. Push to the Dayli server (optional)

In [ ]:
# Push to the Dayli server on Oracle Cloud VPS.
import os

try:
    from google.colab import userdata
    API_KEY = userdata.get("DAYLI_INTERNAL_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = os.environ.get("DAYLI_INTERNAL_API_KEY", "d7f93e1a8b4c2056ae0f89c31274bc95d6e241785f09ab3c8e17429650db1f43").strip(""'")

SERVER_URL = os.environ.get("SERVER_URL", "https://129.159.19.84.sslip.io/internal/aurora/forecast")

if SERVER_URL and API_KEY:
    import base64, urllib.error, urllib.request
    cred = base64.b64encode(f"dayli:{API_KEY}".encode()).decode()
    req = urllib.request.Request(
        SERVER_URL, data=json.dumps(payload).encode(), method="POST",
        headers={"Content-Type": "application/json", "Authorization": f"Basic {cred}"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print("pushed: HTTP", r.status, json.loads(r.read()))
    except urllib.error.HTTPError as e:
        print("push failed: HTTP", e.code, e.read().decode()[:300])
else:
    print("SERVER_URL / API_KEY not set - skipping push")
